# Getting Started

This tutorial will get you up and running with JupyterLab and `pyqualw2`. We'll work through how you can configure, run, and analyze CE-QUAL-W2 simulations directly in the browser with notebooks. Let's get started by installing dependencies if you haven't done so already.

In [ ]:
!pip install -e '..[notebook]'

Next, we import the necessary packages and activate `ipympl`, which allows `matplotlib` graphs to be interactive when displayed in notebooks.

In [ ]:
%matplotlib ipympl

import matplotlib.pyplot as plt
from pathlib import Path

from pyqualw2 import Config, ModelRunner, RunResult, MultiRunResult
from pyqualw2.config.inputs import FlowInput
from pyqualw2.outputs import QWO, QWOLayers

## Configuring and Running Simulations

We start by configuring the simulation. Let's go over the input files needed for `cequalw2` to run:

| Argument | Name | Value | Description |
|------|-----------------|----------|-------------|
| Configuration | `con` | `w2_con.csv` | Main CE-QUAL-W2 configuration file containing simulation parameters and settings |
| Bathymetry | `bathymetry` | `mbth_wb1.csv` | Bathymetry data defining the water body geometry and depth contours |
| Profile | `profile` | `mvpr1.npt` | Initial layer-dependent data for water temperature and other quantities |
| Meteorological Data* | `met_data` | `2018_CEQUAL_met_inputs.csv` | Per-branch meteorological input data including air temperature, solar radiation, wind speed, etc. |
| Shade | `shade` | `mshade.npt` | Shade data for topographic and vegetative shading effects on the water body |
| Wind Sheltering | `wind_sheltering` | `mwsc.npt` | Wind sheltering coefficients accounting for terrain effects on wind |
| Temperature Data* | `temp_data` | `SJA_2018_temp.csv` | Branch temperature inflow data for each model branch |
| Temperature Tributaries* | `temperature_tributaries` | `mtdt_br1.npt` | Temperature data for tributary inflows to each branch |
| Flow Data | `flow_data` | `2018_Observed_Flow.csv` | Combined historical flow data containing inflow, outflow, and evaporation for all branches |
| Flow data date column | `flow_data_date_col` | `"Date"` | Column to use for the date in the flow data |
| Flow data inflow columns* | `flow_data_inflow_cols` | `[["Date", "M_IN"]]` | A list of columns to use for the inflow data _for each branch_. In this simulation we only have one branch, so this is a list with a single set of columns. |
| Flow data outflow columns* | `flow_data_outflow_cols` | `[["Date", "SPL_OUT", "FKC_OUT", "MC_OUT", "SJR_OUT"]]` | A list of columns to use for the outflow data _for each branch_. |
| Flow data evaporation columns* | `flow_data_evaporation_cols` | `[["Date", "MIL_EVAP"]]` | A list of columns to use for the evaporation data _for each branch_. |
| CE-QUAL-W2 Binary | `cequalw2_path` | `w2_v45_64.exe` | The CE-QUAL-W2 executable binary used to run the simulation |

Note that arguments marked with a `*` indicate per-branch quantities which take an option _for each branch in the water model_.

**Notes:**
- The `flow_data` input is optional and can be replaced by separate `branch_inflow`, `branch_outflow`, and `branch_evaporation` files; these are the files directly ingested by `cequalw2`.
- When `flow_data` is provided, it must include column specifications for `flow_data_date_col`, `flow_data_inflow_cols`, `flow_data_outflow_cols`, and `flow_data_evaporation_cols`. The latter three arguments must contain column names _for each branch file_, which is why it is a list of lists here.
- `temp_data` and `temperature_tributaries` are lists that can contain multiple files for different branches

In [ ]:
data_path = Path("../test/sample_data1/")
inputs = data_path / "inputs"
historic = data_path / "historic_data"
output_dir = Path("simple_simulation")

temperature_2018 = historic / "temp_data" / "SJA_2018_temp.csv"
config = Config.from_files(
    name="test_name",
    con=data_path / "w2_con.csv",
    bathymetry=inputs / "mbth_wb1.csv",
    profile=inputs / "mvpr1.npt",
    met_data=historic / "met_data" / "2018_CEQUAL_met_inputs.csv",
    shade=inputs / "mshade.npt",
    temperature_tributaries=[inputs / "mtdt_br1.npt"],
    # See note about why this is 4 separate values, while the other branch-dependent arguments are not.
    temp_data=[temperature_2018, temperature_2018, temperature_2018, temperature_2018],
    flow_data=historic / "flow_data" / "2018_Observed_Flow.csv",
    wind_sheltering=inputs / "mwsc.npt",
    flow_data_date_col="Date",
    flow_data_inflow_cols=[["Date", "M_IN"]],
    flow_data_outflow_cols=[["Date", "SPL_OUT", "FKC_OUT", "MC_OUT", "SJR_OUT"]],
    flow_data_evaporation_cols=[["Date", "MIL_EVAP"]],
    cequalw2_path=Path("../") / "cequalw2" / "w2_v45_64.exe",
)

### An aside about `temp_data`

Note that the `temp_data` argument is specified 4 times: this is needed because the `w2_con.csv` configuration specifies 4 branches even though the Millerton water body system only has 1 branch. Out of an abundance of caution we kept this old configuration as it is, since it is known to work correctly. For this reason we also create dummy flow data for each of the 3 dummy branches:

In [ ]:
# Due to misconfiguration in w2_con.csv for this test data, we need to fiddle
# the branch data to get cequalw2 to run. See
# https://github.com/steelhead-dev/pyqualw2/issues/69 for more information.
# Set all the flow to 0 for branches 2, 3, and 4. Then add three copies of this
# dummy data to serve as those dummy branches
dummy_flow_data = config.branch_inflow[0].data.copy()
dummy_flow_data.iloc[:, 1] = 0
config.branch_inflow.extend(
    [
        FlowInput(data=dummy_flow_data.copy()),
        FlowInput(data=dummy_flow_data.copy()),
        FlowInput(data=dummy_flow_data.copy()),
    ]
)

Once the `Config` is specified, we can create a `ModelRunner` instance to execute `cequalw2` on the `Config` defined above.

<font color='orange'><b>NOTE: </b>CE-QUAL-W2 will exit automatically once it is done running the simulation. There is no need to interact with the UI.</font>

In [ ]:
Runner = ModelRunner(config, output_dir)
Runner.run(overwrite=True)

## Post-Processing and Analysis

To help with data analysis, `pyqualw2` provides output classes which perform some common post-processing and plotting operations on simulation output. Let's look at the flow in each layer as a function of time:

In [ ]:
flow = QWOLayers.from_file("./simple_simulation/test_name/outputs/qwo_layers_31_.csv")
cmap = flow.plot_colormap()

For this example simulation, it looks like most of the flow is happening in a band in the middle of the water body, likely through a reservoir outlet. Interestingly, there's a band of virtually no flow centered on layer ~175, but the layers below that _do_ see some flow. Is it possible that there's a small outlet at the bottom of the water body drawing water from this layer?

## Analyzing Multiple Runs

Let's use `pyqualw2` to analyze multiple run groups at once. Start by loading in the directory to the data:

In [ ]:
mrr = MultiRunResult("../test/sample_multi_run/")
mrr

Let's now make a plot of the composite temperature statistics. This can be done either by manually creating a figure and passing the axes to the `MultiRunResult.plot_two_structure_composite` function:

In [ ]:
fig, ax = plt.subplots(1, 1)
mrr.plot_two_structure_composite(4, ax=ax);

Or if no axes are passed to `MultiRunResult.plot_two_structure_composite`, a figure will be generated for you. Of course, you can also use all the matplotlib tools you normally would as well:

In [ ]:
fig, ax, region, line = mrr.plot_two_structure_composite(4)

# Modify the labels for the min/max region and average temperature line
region.set_label("Min/max temperature range")
line.set_label(r"$T_{avg}$")

# Draw the 14.44°C threshold line
mrr.plot_threshold_line_y(ax, linestyle="--", label="14.44 °C Threshold")

# Add a legend to the plot
ax.legend(loc="best")

# Rotate the x-labels to make it so they don't overlap
xticks, labels = ax.get_xticks(), ax.get_xticklabels()
ax.set_xticks(xticks, labels=labels, rotation=45, ha="right", rotation_mode="anchor")

# Resize the axis labels
ax.set_ylabel(ax.get_ylabel(), fontsize="x-large", labelpad=15)
ax.set_xlabel(ax.get_xlabel(), fontsize="x-large", labelpad=15)

# Resize the figure to accommodate the changes to axis labels
plt.tight_layout()

We can also get the number of days that the minimum, maximum, and average temperatures spent over the threshold:

In [ ]:
mrr.get_days_above_threshold(4, threshold=14.44)

We can also explore the temperature data across all runs for structure 4 with Jupyter's rich support for `pd.DataFrame`:

In [ ]:
run_data, stats = mrr.get_two_structure_composite_data(4)
run_data

In [ ]:
stats